# Customer Churn Prediction with Feedback Mining
**ML + NLP + Data Science lab notebook**

This companion notebook is for deeper experiments in Colab. It evaluates a calibrated churn classifier on a held-out split, mines feedback sentiment and topics, examines cohorts and segments, and scores rows with blank churn labels. It is not a causal retention-uplift model.

Expected input: a CSV with a binary churn target (for example `churn` = 0/1 or Yes/No). Useful optional fields include `customer_id`, `complaint_text`, `tenure_months`, `monthly_charges`, `contract_type`, `signup_date`, support/contact counts and payment-delay counts. Use anonymised data.

In [ ]:
%pip -q install pandas numpy scikit-learn matplotlib vaderSentiment

## 1. Upload a CSV
Upload your customer table. The repository includes `sample_data/demo_customers.csv` and `public/sample-churn-data.csv`, both synthetic. In Colab, download one of those files from GitHub or from the project ZIP, then upload it below.

In [ ]:
from google.colab import files
import io, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

uploaded = files.upload()
if not uploaded:
    raise ValueError('Please upload one CSV file to begin.')
csv_name = next(name for name in uploaded if name.lower().endswith('.csv'))
df = pd.read_csv(io.BytesIO(uploaded[csv_name]))
df.columns = [str(c).strip() for c in df.columns]
print(f'Loaded {csv_name}: {df.shape[0]:,} rows × {df.shape[1]} columns')
display(df.head())
print('Columns:', df.columns.tolist())

## 2. Configure columns and inspect data
The following guesses common names. **Check and change the configuration** to match your file. Leave `text_col`, `id_col`, or `date_col` as `None` if unavailable.

In [ ]:
def find_col(columns, exact=(), contains=()):
    normal = {re.sub(r'[^a-z0-9]+', '_', str(c).lower()).strip('_'): c for c in columns}
    for candidate in exact:
        if candidate in normal: return normal[candidate]
    for key, value in normal.items():
        if any(t in key for t in contains): return value
    return None

target_col = find_col(df.columns, ('churn','churned','is_churn','exited','attrition','left','target'), ('churn','attrition','exited'))
text_col = find_col(df.columns, ('complaint_text','feedback_text','customer_feedback','review_text','text'), ('complaint','feedback','review','comment','message','text'))
id_col = find_col(df.columns, ('customer_id','customerid','account_id','user_id','id'), ('customer_id','account_id','user_id'))
date_col = find_col(df.columns, ('signup_date','join_date','start_date','created_at','cohort_date'), ('signup_date','join_date','created_at','start_date'))
print({'target_col': target_col, 'text_col': text_col, 'id_col': id_col, 'date_col': date_col})

# Edit these values explicitly if automatic column detection is wrong.
# target_col = 'Churn'
# text_col = 'Customer complaint'

display(df.isna().mean().sort_values(ascending=False).head(15).rename('missing_fraction').to_frame())
if target_col is None: raise ValueError('Set target_col to the binary churn outcome column.')
display(df[target_col].value_counts(dropna=False).rename('count').to_frame())

## 3. Encode the churn target and explore the dataset
Blank target labels are held out from training and remain eligible for scoring. For text labels, the code recognises common Yes/No, Churn/Retained and Active/Inactive names. Review the positive-class mapping before interpreting metrics.

In [ ]:
raw_target = df[target_col].copy()
if raw_target.dtype == object or pd.api.types.is_string_dtype(raw_target):
    raw_target = raw_target.astype('string').str.strip().replace({'': pd.NA, 'NA': pd.NA, 'N/A': pd.NA, 'null': pd.NA})
labelled_mask = raw_target.notna()
label_values = raw_target[labelled_mask]
if label_values.nunique() != 2: raise ValueError(f'Target must have two observed labels; got {label_values.nunique()}.')
numeric_labels = pd.to_numeric(label_values, errors='coerce')
if numeric_labels.notna().all():
    vals = sorted(numeric_labels.unique().tolist())
    positive_value = 1 if vals == [0, 1] else vals[-1]
    y = pd.Series(np.nan, index=df.index, dtype=float)
    y.loc[labelled_mask] = (numeric_labels == positive_value).astype(int).to_numpy()
    positive_label = str(positive_value)
else:
    text_labels = label_values.astype(str).str.lower().str.strip()
    unique = sorted(text_labels.unique().tolist())
    positives = {'yes','y','true','churn','churned','left','exited','cancelled','canceled','lost','inactive','1'}
    negatives = {'no','n','false','stay','stayed','retained','active','0','not churn','not_churn'}
    p = next((v for v in unique if v in positives or ('churn' in v and not v.startswith('not '))), None)
    n = next((v for v in unique if v in negatives or v.startswith('not churn')), None)
    if p is None and n is None: n, p = unique[0], unique[1]
    elif p is None: p = next(v for v in unique if v != n)
    elif n is None: n = next(v for v in unique if v != p)
    y = pd.Series(np.nan, index=df.index, dtype=float)
    y.loc[labelled_mask] = (text_labels == p).astype(int).to_numpy()
    positive_label = p

labelled_y = y.dropna().astype(int)
print('Positive/churn label:', positive_label)
print('Labelled rows:', len(labelled_y), '| unlabelled rows to score:', int(y.isna().sum()))
print('Churn rate among labelled rows:', round(float(labelled_y.mean()), 4))
if len(labelled_y) < 50 or labelled_y.value_counts().min() < 12:
    raise ValueError('Use at least 50 labelled records with at least 12 examples per class for this stratified calibration workflow.')
ax = labelled_y.value_counts().sort_index().plot(kind='bar', color=['#8aa58d','#315d46'], title='Labelled churn outcome counts')
ax.set_xlabel('Encoded label (0 = retained, 1 = churn)'); ax.set_ylabel('Customers'); plt.show()

## 4. Build text + structured features
Text uses TF-IDF n-grams. Numeric features are median-imputed and categorical features are imputed then one-hot encoded. IDs, dates and the target are excluded from predictors. Make sure no feature was created after the churn decision (which would leak the answer).

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score, precision_score, recall_score, f1_score, brier_score_loss)

exclude = {target_col}
if id_col: exclude.add(id_col)
if date_col: exclude.add(date_col)
exclude |= {c for c in df.columns if str(c).lower().strip() in {'email','email_address','phone','phone_number','full_name','address','postcode','zip_code'}}
feature_cols = [c for c in df.columns if c not in exclude]
X_all = df[feature_cols].copy()
if text_col in X_all.columns if text_col else False:
    X_all[text_col] = X_all[text_col].fillna('').astype(str).replace('', 'no feedback provided')
empty_cols = [c for c in X_all.columns if X_all[c].isna().all()]
X_all = X_all.drop(columns=empty_cols)
if text_col in empty_cols if text_col else False: text_col = None
numeric_cols = [c for c in X_all.columns if c != text_col and pd.api.types.is_numeric_dtype(X_all[c])][:18]
categorical_cols = [c for c in X_all.columns if c != text_col and not pd.api.types.is_numeric_dtype(X_all[c])][:18]
transformers = []
if text_col and text_col in X_all.columns:
    transformers.append(('text', TfidfVectorizer(max_features=500, ngram_range=(1,2), min_df=1, sublinear_tf=True), text_col))
if numeric_cols:
    transformers.append(('numeric', Pipeline([('imputer', SimpleImputer(strategy='median', add_indicator=True)), ('scale', StandardScaler())]), numeric_cols))
if categorical_cols:
    transformers.append(('categorical', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore', max_categories=25))]), categorical_cols))
if not transformers: raise ValueError('No usable feature columns; check the target, ID and date columns.')
preprocessor = ColumnTransformer(transformers, remainder='drop', sparse_threshold=0.25)
X_labelled = X_all.loc[labelled_mask.to_numpy()]
y_labelled = labelled_y
X_train, X_test, y_train, y_test = train_test_split(X_labelled, y_labelled, test_size=0.25, random_state=42, stratify=y_labelled)
print(f'Train: {len(X_train):,}; held-out test: {len(X_test):,}; scoring rows: {len(X_all):,}')

## 5. Train, calibrate, and measure unseen performance
The holdout set is used only for evaluation. Logistic Regression probabilities are calibrated with sigmoid calibration. Random Forest is a comparison baseline; the raw RF probabilities are not calibrated in this notebook cell.

In [ ]:
def new_pipeline(pre, estimator):
    return Pipeline([('features', pre), ('classifier', estimator)])

logit_pipe = new_pipeline(preprocessor, LogisticRegression(max_iter=500, class_weight='balanced', solver='liblinear', random_state=42))
calibrated_model = CalibratedClassifierCV(estimator=logit_pipe, method='sigmoid', cv=3)
calibrated_model.fit(X_train, y_train)
p_cal = calibrated_model.predict_proba(X_test)[:, 1]
rf_pre = ColumnTransformer(transformers, remainder='drop', sparse_threshold=0.25)
rf_model = new_pipeline(rf_pre, RandomForestClassifier(n_estimators=55, max_depth=9, min_samples_leaf=3, class_weight='balanced_subsample', random_state=42, n_jobs=1))
rf_model.fit(X_train, y_train)
p_rf = rf_model.predict_proba(X_test)[:, 1]

def metric_row(name, y_true, prob, calibration_label):
    pred = (prob >= .5).astype(int)
    return {'Model': name, 'Calibration': calibration_label, 'ROC AUC': roc_auc_score(y_true, prob), 'Avg precision': average_precision_score(y_true, prob), 'Accuracy': accuracy_score(y_true, pred), 'Precision': precision_score(y_true, pred, zero_division=0), 'Recall': recall_score(y_true, pred, zero_division=0), 'F1': f1_score(y_true, pred, zero_division=0), 'Brier': brier_score_loss(y_true, prob)}

metrics_df = pd.DataFrame([metric_row('Calibrated Logistic Regression', y_test, p_cal, 'Sigmoid calibrated'), metric_row('Random Forest baseline', y_test, p_rf, 'Raw probabilities')])
display(metrics_df.round(3))
print('Interpretation: test-set metrics estimate performance on unseen rows from a similar distribution; they are not a guarantee of future performance.')

## 6. Probability calibration curve
A reliable 70% probability bucket should have about 70% observed churn over enough similar cases. Small samples make bucket rates noisy.

In [ ]:
from sklearn.calibration import calibration_curve
fraction_positive, mean_predicted = calibration_curve(y_test, p_cal, n_bins=8, strategy='quantile')
plt.figure(figsize=(5.8, 5.2))
plt.plot([0,1], [0,1], '--', color='#b8c5b6', label='Perfect calibration')
plt.plot(mean_predicted, fraction_positive, marker='o', color='#32694a', label='Calibrated Logistic Regression')
plt.xlabel('Mean predicted churn probability'); plt.ylabel('Observed churn fraction'); plt.title('Holdout calibration curve'); plt.legend(); plt.grid(alpha=.2); plt.show()

## 7. Complaint sentiment and topic extraction
VADER is a lightweight lexicon-based estimate (best for English text). NMF topics are corpus-dependent groups of recurring TF-IDF terms; review topic terms before naming them in a report.

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

if text_col and text_col in df.columns:
    texts = df[text_col].fillna('').astype(str).str.replace(r'\s+', ' ', regex=True).str.strip()
    analyzer = SentimentIntensityAnalyzer()
    df['feedback_sentiment_score'] = texts.map(lambda t: analyzer.polarity_scores(t)['compound'] if t else 0.0)
    df['feedback_sentiment'] = np.where(df['feedback_sentiment_score'] <= -.05, 'Negative', np.where(df['feedback_sentiment_score'] >= .05, 'Positive', 'Neutral'))
    display(df['feedback_sentiment'].value_counts().rename_axis('sentiment').to_frame('customers'))
    valid_text = texts.str.len() > 0
    topic_terms = []
    if valid_text.sum() >= 12 and texts[valid_text].nunique() >= 5:
        try:
            topic_vec = TfidfVectorizer(stop_words='english', max_features=350, ngram_range=(1,2), min_df=2, max_df=.95)
            topic_matrix = topic_vec.fit_transform(texts[valid_text])
            if topic_matrix.shape[1] >= 4:
                n_topics = min(5, max(2, int(valid_text.sum()) // 35), topic_matrix.shape[1])
                nmf = NMF(n_components=n_topics, init='nndsvda', max_iter=120, random_state=42)
                topic_docs = nmf.fit_transform(topic_matrix)
                terms = topic_vec.get_feature_names_out()
                topic_labels = []
                for i, weights in enumerate(nmf.components_):
                    top = terms[weights.argsort()[-5:][::-1]].tolist()
                    topic_terms.append({'topic': f'Topic {i+1}', 'terms': top})
                    topic_labels.append(' · '.join(top[:3]))
                topic_result = pd.Series(pd.NA, index=df.index, dtype='string')
                topic_result.loc[valid_text] = [topic_labels[int(i)] for i in topic_docs.argmax(axis=1)]
                df['feedback_topic'] = topic_result
        except ValueError as exc:
            print('Topic extraction skipped:', exc)
    print('Top topic terms:', topic_terms)
else:
    print('No complaint text column selected; skipping sentiment and topics.')

## 8. Cohorts and numeric customer segments
Signup month or tenure bands can show group-level churn. KMeans is run on available numeric features and scaled first. Cluster labels are arbitrary IDs, not ranked value judgments.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

df['actual_churn_encoded'] = y
if date_col and date_col in df.columns:
    df['cohort'] = pd.to_datetime(df[date_col], errors='coerce').dt.to_period('M').astype('string')
elif any('tenure' in str(c).lower() for c in df.columns):
    tenure_column = next(c for c in df.columns if 'tenure' in str(c).lower())
    df['cohort'] = pd.cut(pd.to_numeric(df[tenure_column], errors='coerce'), [-np.inf,3,6,12,24,36,60,np.inf], labels=['≤3 mo','4–6 mo','7–12 mo','13–24 mo','25–36 mo','37–60 mo','>60 mo']).astype('string')
else:
    df['cohort'] = pd.NA
cohort_table = df[df['actual_churn_encoded'].notna()].groupby('cohort', dropna=True, observed=True)['actual_churn_encoded'].agg(customers='count', churn_rate='mean').reset_index()
cohort_table['retention_rate'] = 1 - cohort_table['churn_rate']
display(cohort_table.tail(18).round(3))

cluster_cols = [c for c in numeric_cols if pd.to_numeric(X_all[c], errors='coerce').nunique(dropna=True) > 1][:6]
if len(cluster_cols) >= 2 and len(df) >= 18:
    cluster_values = SimpleImputer(strategy='median').fit_transform(X_all[cluster_cols].apply(pd.to_numeric, errors='coerce'))
    cluster_scaled = StandardScaler().fit_transform(cluster_values)
    n_clusters = min(4, max(2, len(df)//50))
    df['customer_segment'] = [f'Segment {i+1}' for i in KMeans(n_clusters=n_clusters, n_init=10, random_state=42).fit_predict(cluster_scaled)]
    segment_table = df.groupby('customer_segment').agg(customers=(target_col, 'size'), average_tenure=(cluster_cols[0], 'mean')).reset_index()
    display(segment_table.round(2))
else:
    print('Not enough varied numeric features for KMeans segmentation.')

## 9. Rank retention actions and score every customer
The action list below uses rules such as negative feedback, billing terms, repeat support contacts and elevated calibrated risk. These are operational hypotheses, not learned treatment effects. First inspect the model scores and text signals, then test action effectiveness with an experiment.

In [ ]:
# Refit the calibrated model on all labelled historical customers for scoring.# Holdout metrics above remain the reported estimate of unseen performance.
final_model = CalibratedClassifierCV(estimator=new_pipeline(preprocessor, LogisticRegression(max_iter=500, class_weight='balanced', solver='liblinear', random_state=42)), method='sigmoid', cv=3)
final_model.fit(X_labelled, y_labelled)
df['churn_probability'] = final_model.predict_proba(X_all)[:, 1]
df['risk_tier'] = pd.cut(df['churn_probability'], bins=[-0.001,.30,.55,.75,1.001], labels=['Low','Medium','High','Critical']).astype('string')

def recommended_action(row):
    p = float(row['churn_probability'])
    text = str(row.get(text_col, '') if text_col else '').lower()
    if any(word in text for word in ('bill','billing','price','expensive','cost','charge','fee','payment')):
        return 'Billing and plan review'
    if any(word in text for word in ('support','agent','wait','ticket','response','unresolved')):
        return 'Dedicated support follow-up'
    if any(word in text for word in ('outage','slow','broken','error','crash')):
        return 'Product reliability follow-up'
    if p >= .55:
        return 'Proactive retention outreach'
    return 'Review account health'

df['recommended_action'] = df.apply(recommended_action, axis=1)
priority_actions = (df.assign(_priority=df['churn_probability']*100).groupby('recommended_action').agg(customers=('recommended_action','size'), average_risk=('churn_probability','mean'), priority_total=('_priority','sum')).sort_values('priority_total', ascending=False).reset_index())
display(priority_actions.round(3))
display(df.sort_values('churn_probability', ascending=False)[[c for c in [id_col, target_col, 'churn_probability','risk_tier'] if c and c in df.columns] + ['recommended_action']].head(20))

## 10. Export the customer worklist
The exported CSV includes predicted risks and action suggestions. Rows whose actual churn target is blank were not used for model fitting or holdout evaluation, but were still scored by the final model.

In [ ]:
output_path = 'churn_feedback_predictions.csv'
df.to_csv(output_path, index=False)
print('Saved', output_path, '| rows:', len(df))
files.download(output_path)